# Imports

In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load data

In [ ]:
appart_df = pd.read_csv('datasets_prepd/dvf_appartement.csv')
print(appart_df.head())

# Features

In [ ]:
features = [
    "longitude",
    "latitude",
    "code_postal",
    "surface_reelle_bati",
    "nombre_pieces_principales",
    "total_carrez_surface",
    "number_of_lots",
    "season"
]

target = "valeur_fonciere_actualisee"

# Encode season to numeric values
season_mapping = {"winter": 0, "spring": 1, "summer": 2, "autumn": 3}
appart_df["season"] = appart_df["season"].map(season_mapping)

X = appart_df[features]
y = appart_df[target]

print("Feature count:", X.shape[1])
print(X.head())
print(y.head())

# Make sets : trail and test and validate

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print("Train size:", X_train.shape)
print("Validation size:", X_val.shape)
print("Test size:", X_test.shape)

# Ssave

In [ ]:
os.makedirs("data_apt", exist_ok=True)

train_df = X_train.copy()
val_df = X_val.copy()
test_df = X_test.copy()

train_df[target] = y_train
val_df[target] = y_val
test_df[target] = y_test

train_df.to_csv("data_apt/appart_train.csv", index=False)
val_df.to_csv("data_apt/appart_val.csv", index=False)
test_df.to_csv("data_apt/appart_test.csv", index=False)

print("Datasets saved.")

Datasets saved.


# Train model

In [ ]:
model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=4,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
print("Model trained.")

# Feature importance

In [ ]:
importances = model.feature_importances_
feat_imp = pd.Series(importances, index=features).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
feat_imp.plot(kind='barh')
plt.title("Feature Importance - Appartements")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

print("\nFeature importances:")
for name, imp in feat_imp.sort_values(ascending=False).items():
    print(f"  {name}: {imp:.4f}")

In [ ]:
# Evaluation

In [ ]:
# Evaluate on all 3 sets
for name, X_set, y_set in [("Train", X_train, y_train), ("Validation", X_val, y_val), ("Test", X_test, y_test)]:
    y_pred = model.predict(X_set)
    rmse = np.sqrt(mean_squared_error(y_set, y_pred))
    mae = mean_absolute_error(y_set, y_pred)
    r2 = r2_score(y_set, y_pred)
    erreur_pct = (abs(y_set - y_pred) / y_set * 100).mean()
    print(f"{name:10s} — RMSE: {rmse:>10.0f} | MAE: {mae:>10.0f} | R2: {r2:.4f} | Erreur moy: {erreur_pct:.2f}%")

# Overfitting gap
r2_train = r2_score(y_train, model.predict(X_train))
r2_val = r2_score(y_val, model.predict(X_val))
gap = r2_train - r2_val
print(f"\nGap surapprentissage (Train R2 - Val R2): {gap:.4f}")

# Comparaison GradientBoosting

In [ ]:
gb = HistGradientBoostingRegressor(
    max_iter=300, max_depth=6, learning_rate=0.1,
    min_samples_leaf=10, random_state=42
)
gb.fit(X_train, y_train)

for name, X_set, y_set in [("Train", X_train, y_train), ("Validation", X_val, y_val), ("Test", X_test, y_test)]:
    y_pred = gb.predict(X_set)
    r2 = r2_score(y_set, y_pred)
    erreur_pct = (abs(y_set - y_pred) / y_set * 100).mean()
    print(f"GB {name:10s} — R2: {r2:.4f} | Erreur moy: {erreur_pct:.2f}%")

gb_gap = r2_score(y_train, gb.predict(X_train)) - r2_score(y_val, gb.predict(X_val))
print(f"\nGB Gap surapprentissage: {gb_gap:.4f}")

GB Train      — R2: 0.8133 | Erreur moy: 13.31%
GB Validation — R2: 0.7618 | Erreur moy: 14.69%
GB Test       — R2: 0.7619 | Erreur moy: 14.73%

GB Gap surapprentissage: 0.0514


In [ ]:
SAVE MODEL

In [ ]:
y_test_pred = model_appart.predict(X_test)

rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae_test = mean_absolute_error(y_test, y_test_pred)
r2_test = r2_score(y_test, y_test_pred)

print("Test RMSE:", rmse_test)
print("Test MAE:", mae_test)
print("Test R2:", r2_test)

In [ ]:
df_test_eval = pd.DataFrame({
    "prix_reel": y_test,
    "prix_predit": y_test_pred
})

df_test_eval["erreur_pct"] = abs(df_test_eval["prix_reel"] - df_test_eval["prix_predit"]) / df_test_eval["prix_reel"] * 100

print("Erreur moyenne test (%) :", df_test_eval["erreur_pct"].mean())